# Module 04: Confusion Matrices & Macro-F1

**ECBS5200 Pre-Work** — Practical Deep Learning Engineering

In this notebook you'll see why accuracy is a misleading metric for
imbalanced classification, learn to read confusion matrices, and
understand why macro-F1 is our primary metric for the course.

**No GPU needed** — everything here runs on CPU in under a minute.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
    accuracy_score,
    f1_score,
)

np.random.seed(42)

## Part 1: A small synthetic example

Let's build a tiny 5-class problem with deliberate class imbalance.
Class 0 is the dominant class with 50 examples; the other four classes
have only 5-15 examples each. This mirrors (in miniature) what our
real complaint dataset looks like (153 raw categories, reduced to 113 in Week 1).

In [ ]:
# Ground-truth labels: heavily imbalanced
y_true = np.array(
    [0] * 50  # Class 0: 50 examples (dominant)
    + [1] * 15  # Class 1: 15 examples
    + [2] * 15  # Class 2: 15 examples
    + [3] * 10  # Class 3: 10 examples
    + [4] * 10  # Class 4: 10 examples (rare)
)

class_names = ["Billing", "Fraud", "Mortgage", "Student Loan", "Crypto"]

print("Class distribution:")
for i, name in enumerate(class_names):
    count = np.sum(y_true == i)
    print(f"  {name}: {count} examples ({count / len(y_true) * 100:.0f}%)")
print(f"\nTotal: {len(y_true)} examples")

### The "majority class predictor"

This is the dumbest possible model: always predict the most common class.

In [ ]:
# Strategy 1: always predict the majority class
y_pred_majority = np.zeros_like(y_true)  # Always predict class 0

acc_majority = accuracy_score(y_true, y_pred_majority)
f1_majority = f1_score(y_true, y_pred_majority, average="macro", zero_division=0)

print("=== Majority-Class Predictor ===")
print(f"Accuracy:  {acc_majority:.1%}")
print(f"Macro-F1:  {f1_majority:.3f}")
print()
print("Accuracy says 'not bad.' Macro-F1 says 'this model is useless.'")

### A "decent" model

Now let's simulate a model that actually tries — it gets most things
right but makes realistic mistakes.

In [ ]:
# Strategy 2: a model that actually tries (simulated predictions)
rng = np.random.default_rng(42)
y_pred_decent = y_true.copy()

# Introduce ~20% errors: randomly flip some predictions
n_errors = int(len(y_true) * 0.20)
error_indices = rng.choice(len(y_true), size=n_errors, replace=False)
for idx in error_indices:
    wrong_classes = [c for c in range(5) if c != y_true[idx]]
    y_pred_decent[idx] = rng.choice(wrong_classes)

acc_decent = accuracy_score(y_true, y_pred_decent)
f1_decent = f1_score(y_true, y_pred_decent, average="macro")

print("=== Decent Model ===")
print(f"Accuracy:  {acc_decent:.1%}")
print(f"Macro-F1:  {f1_decent:.3f}")

### Confusion matrix for the decent model

Rows = true labels, columns = predicted labels. The diagonal shows
correct predictions. Off-diagonal entries show specific confusions.

In [ ]:
cm = confusion_matrix(y_true, y_pred_decent)

fig, ax = plt.subplots(figsize=(7, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(ax=ax, cmap="Blues", colorbar=True)
ax.set_title("Confusion Matrix — Decent Model (5 classes)", fontsize=13)
plt.tight_layout()
plt.show()

### Reading the confusion matrix

Look at the off-diagonal entries. Each one tells you: "the model
predicted column-class when the truth was row-class." Large
off-diagonal values reveal specific confusion patterns — classes the
model can't tell apart.

In [ ]:
print("=== Per-class classification report ===\n")
print(classification_report(y_true, y_pred_decent, target_names=class_names))

### Accuracy vs Macro-F1: side by side

In [ ]:
print("=== Comparison ===")
print(f"{'Model':<25} {'Accuracy':>10} {'Macro-F1':>10}")
print("-" * 47)
print(f"{'Majority-class predictor':<25} {acc_majority:>10.1%} {f1_majority:>10.3f}")
print(f"{'Decent model':<25} {acc_decent:>10.1%} {f1_decent:>10.3f}")
print()
print("The majority predictor has reasonable accuracy but near-zero macro-F1.")
print("Macro-F1 correctly identifies that it has learned nothing useful.")

## Part 2: The real dataset

Now let's look at the actual consumer complaints dataset we'll use all
semester. We want to see just how imbalanced it is.

In [ ]:
from datasets import load_dataset

dataset = load_dataset("determined-ai/consumer_complaints_medium", split="train")
print(f"Total complaints: {len(dataset):,}")
print(f"Columns: {dataset.column_names}")

In [ ]:
# Count examples per class
from collections import Counter

issue_counts = Counter(dataset["Issue"])
print(f"\nNumber of unique classes: {len(issue_counts)}")

# Sort by frequency
sorted_issues = issue_counts.most_common()
print(f"\nTop 5 classes:")
for issue, count in sorted_issues[:5]:
    print(f"  {issue}: {count:,} ({count / len(dataset) * 100:.1f}%)")

print(f"\nBottom 5 classes:")
for issue, count in sorted_issues[-5:]:
    print(f"  {issue}: {count:,} ({count / len(dataset) * 100:.2f}%)")

**Notice something odd?** The top two classes are "Incorrect information
on credit report" and "Incorrect information on your report." Those sound
like the same thing — and they basically are. The dataset has some
near-duplicate categories from a form change in 2017. We'll clean
those up in Week 1, merging 153 raw categories down to 113. For now,
the important thing is the massive imbalance.

### Visualizing the imbalance

Let's plot the top 20 classes vs the bottom 20. The scale difference
makes the imbalance visceral.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Top 20
top_20 = sorted_issues[:20]
top_labels = [issue[:40] for issue, _ in top_20]
top_counts = [count for _, count in top_20]

axes[0].barh(range(len(top_20)), top_counts, color="steelblue")
axes[0].set_yticks(range(len(top_20)))
axes[0].set_yticklabels(top_labels, fontsize=8)
axes[0].invert_yaxis()
axes[0].set_xlabel("Number of examples")
axes[0].set_title("Top 20 classes (most examples)", fontsize=12)

# Bottom 20
bottom_20 = sorted_issues[-20:]
bottom_labels = [issue[:40] for issue, _ in bottom_20]
bottom_counts = [count for _, count in bottom_20]

axes[1].barh(range(len(bottom_20)), bottom_counts, color="coral")
axes[1].set_yticks(range(len(bottom_20)))
axes[1].set_yticklabels(bottom_labels, fontsize=8)
axes[1].invert_yaxis()
axes[1].set_xlabel("Number of examples")
axes[1].set_title("Bottom 20 classes (fewest examples)", fontsize=12)

plt.suptitle("Class imbalance in the consumer complaints dataset", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

# Print the ratio
print(f"\nLargest class:  {sorted_issues[0][1]:,} examples")
print(f"Smallest class: {sorted_issues[-1][1]:,} examples")
print(f"Ratio: {sorted_issues[0][1] / sorted_issues[-1][1]:.0f}x difference")

### Majority-class predictor on the real data

What happens if we just predict the most common class for every
complaint in the dataset?

In [ ]:
# Find the majority class
majority_class = sorted_issues[0][0]
print(f"Majority class: '{majority_class}'")
print(f"  {sorted_issues[0][1]:,} out of {len(dataset):,} examples "
      f"({sorted_issues[0][1] / len(dataset) * 100:.1f}%)\n")

# Build "predictions": always the majority class
all_labels = dataset["Issue"]
y_true_real = all_labels
y_pred_majority_real = [majority_class] * len(all_labels)

acc_real = accuracy_score(y_true_real, y_pred_majority_real)
f1_real = f1_score(
    y_true_real, y_pred_majority_real, average="macro", zero_division=0
)

print(f"=== Majority-Class Predictor on Real Data ===")
print(f"Accuracy:  {acc_real:.1%}")
print(f"Macro-F1:  {f1_real:.4f}")
print()
print(f"Accuracy: '{acc_real:.1%}' — better than random ({1/len(issue_counts):.2%})")
print(f"Macro-F1: '{f1_real:.4f}' — essentially zero")
print()
print("This model gets exactly ONE class right and completely fails on the")
print(f"other {len(issue_counts) - 1}. Macro-F1 catches this. Accuracy doesn't.")

## Check yourself

Before moving on, make sure you can answer these:

1. **Why is accuracy misleading for imbalanced classification?**
   It's dominated by whatever classes have the most examples. A model
   can get high accuracy by ignoring rare classes entirely.

2. **What does an off-diagonal entry in a confusion matrix tell you?**
   It tells you the model confused one class for another — specifically,
   how many times examples of the row-class were predicted as the
   column-class.

3. **What's the difference between precision and recall?**
   Precision: "when the model predicts this class, how often is it right?"
   Recall: "of all examples of this class, how many did the model find?"

4. **Why do we use macro-F1 instead of accuracy?**
   Macro-F1 treats every class equally regardless of size. With 100+
   classes of very different frequencies, this ensures the model must
   perform well on rare classes too — not just the popular ones.

5. **What would a perfect macro-F1 score be? What would a terrible one be?**
   Perfect = 1.0 (every class has F1 = 1.0). Terrible = 0.0.